# 에이전트 평가 세 전략

이 노트북은 Python 3.12 새 커널에서 독립 실행합니다.

## 패키지 설치

In [1]:
# 노트북이 사용하는 패키지 버전을 새 커널 환경에 설치합니다.
%pip install -q agentevals==0.0.9 langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 환경 준비

### 코드 읽기와 비교 경계

평가 단위는 평가 함수가 입력에서 어느 부분을 읽는지 정합니다. 최종 응답 평가는 마지막 `AIMessage.text`만 읽고, 한 단계 평가는 지정한 `tool_calls`의 이름이나 인자를 읽으며, 궤적 평가는 사용자 메시지부터 모든 AI·도구 메시지의 순서를 읽습니다. `trace()`는 실제 에이전트를 실행하는 함수가 아니라 비교용 메시지 목록을 만드는 보조 함수입니다. 따라서 이 절의 결과는 평가 전략의 차이를 재현하는 고정 예시입니다.

같은 실패도 평가 단위에 따라 보이거나 숨습니다. 도구를 네 번 호출해도 마지막 답이 맞으면 최종 응답 평가는 통과할 수 있습니다. 첫 도구가 맞아도 두 번째 단계에서 권한 없는 도구를 사용하면 한 단계 평가만으로는 놓칩니다. 반대로 궤적이 기준과 같아도 규정 원문이 오래되었다면 사실 오류는 남습니다. 운영 평가에서는 답의 사실성, 필수 단계, 금지 행동, 호출 효율을 각각 측정하고 어느 하나를 전체 품질 점수로 대신하지 않습니다.


In [2]:
# 공개하지 않은 API 키는 존재 여부만 확인하고 고정한 패키지 버전을 출력합니다.
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
print("agentevals", version("agentevals"))
print("langchain", version("langchain"))

agentevals 0.0.9
langchain 1.4.0


## 기준 궤적 준비

In [3]:
# search_rules 다음 check_source가 이어지는 기준 메시지 궤적을 만듭니다.
from langchain.messages import HumanMessage, AIMessage, ToolMessage
def trace(*tools: str) -> list:
    messages=[HumanMessage("규정을 확인해 주세요.")]
    for i,name in enumerate(tools,1):
        messages += [AIMessage("", tool_calls=[{"name":name,"args":{"topic":"대출"},"id":f"c{i}","type":"tool_call"}]), ToolMessage("확인", tool_call_id=f"c{i}")]
    messages.append(AIMessage("일반 도서는 14일입니다."))
    return messages
reference=trace("search_rules", "check_source")

## 세 종류의 평가 입력 만들기

In [4]:
# 같은 메시지 궤적에서 최종 답, 첫 도구 선택, 전체 호출 순서를 각각 추출합니다.
actual=trace("search_rules", "check_source")
final_response = "14일" in actual[-1].text
single_step = actual[1].tool_calls[0]["name"] == "search_rules"
trajectory = [call["name"] for m in actual for call in getattr(m,"tool_calls",[])]
print({"final_response":final_response,"single_step":single_step,"trajectory":trajectory})

{'final_response': True, 'single_step': True, 'trajectory': ['search_rules', 'check_source']}


## 최종 답만 맞는 나쁜 경로 비교

In [5]:
# 답은 유지하고 검색만 반복해 Final Response 평가가 놓치는 호출 증가를 만듭니다.
wasteful=trace("search_rules","search_rules","search_rules","check_source")
print("final pass:", "14일" in wasteful[-1].text)
print("reference calls:", 2, "wasteful calls:", sum(len(getattr(m,"tool_calls",[])) for m in wasteful))
assert "14일" in wasteful[-1].text and len(wasteful)>len(reference)

final pass: True
reference calls: 2 wasteful calls: 4
